In [0]:
# DBTITLE 1,Setup: REST API auth pakai notebook context
import requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}"}

def get_pipeline_spec(pipeline_id):
    r = requests.get(f"{host}/api/2.0/pipelines/{pipeline_id}", headers=headers)
    r.raise_for_status()
    return r.json()

In [0]:
# DBTITLE 1,Cek konfigurasi event log tiap pipeline
pipeline_info = []

for pid in pipeline_ids:
    try:
        resp = get_pipeline_spec(pid)

        # struktur response bisa nested di "spec" atau top-level, kita cek dua-duanya
        spec = resp.get("spec", resp)

        name = spec.get("name", pid)
        catalog = spec.get("catalog")
        schema = spec.get("schema")
        event_log_cfg = spec.get("event_log")

        if event_log_cfg:
            elog_catalog = event_log_cfg.get("catalog") or catalog
            elog_schema = event_log_cfg.get("schema") or schema
            elog_table = event_log_cfg.get("name", "event_log")
            full_table = f"{elog_catalog}.{elog_schema}.{elog_table}"
            status = "PUBLISHED"
        else:
            full_table = None
            status = "NOT_PUBLISHED"

        pipeline_info.append({
            "pipeline_id": pid,
            "name": name,
            "catalog": catalog,
            "schema": schema,
            "event_log_status": status,
            "event_log_table": full_table
        })
    except Exception as e:
        pipeline_info.append({
            "pipeline_id": pid,
            "name": None,
            "catalog": None,
            "schema": None,
            "event_log_status": f"ERROR: {e}",
            "event_log_table": None
        })

# tampilkan hasilnya
import pandas as pd
info_df = pd.DataFrame(pipeline_info)
display(spark.createDataFrame(info_df))

not_published = [p for p in pipeline_info if p["event_log_status"] == "NOT_PUBLISHED"]
print(f"\n{len(not_published)} dari {len(pipeline_info)} pipeline belum publish event log ke UC table.")